# IO Cloud Agent Cloud — MCP Tutorial

## What is Agent Cloud?

[io.net](https://io.net) provides an **MCP (Model Context Protocol) server** that allows AI agents (Claude Code, Cursor, Windsurf, etc.) to manage decentralized GPU infrastructure directly through natural language.

**What you will learn in this tutorial:**

| Step | Feature | Cost |
|------|---------|------|
| 1 | Connect to the MCP server and list all available tools | Free |
| 2 | Browse the CaaS hardware catalog | Free |
| 3 | Estimate deployment price | Free |
| 4 | **Deploy the cheapest container** | **Costs money** |
| 5 | Check deployment status & container details | Free |
| 6 | List all deployments | Free |
| 7 | **Destroy the deployment (save money!)** | Free |

> Note: Step 4 incurs real charges. We will choose the cheapest configuration (1 GPU, 1 replica, 1 hour) and destroy it promptly at the end.

## 0. Environment Setup

In [ ]:
import os
# Uncomment the following two lines if you need a proxy to access the internet
# os.environ['http_proxy']  = 'http://127.0.0.1:7890'
# os.environ['https_proxy'] = 'http://127.0.0.1:7890'

In [ ]:
import asyncio, json, time
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# ============================================================
# Enter your IO Cloud MCP API key here
# Use your io.net Cloud key (requires io-cloud project permissions)
# ============================================================
API_KEY = 'io-v2-*****'

MCP_URL = 'https://mcp.io.solutions/mcp'
HEADERS = {'x-api-key': API_KEY}

print('OK')

### Helper Functions

In [ ]:
async def call_tool(tool_name, arguments=None, retries=3):
    """Connect to the MCP server and call the specified tool, with retries."""
    for attempt in range(retries):
        try:
            async with streamablehttp_client(MCP_URL, headers=HEADERS, timeout=60) as (r, w, _):
                async with ClientSession(r, w) as session:
                    await session.initialize()
                    result = await session.call_tool(tool_name, arguments=arguments or {})
                    text = result.content[0].text if result.content else ''
                    try:
                        return json.loads(text)
                    except json.JSONDecodeError:
                        return text
        except Exception as e:
            if attempt < retries - 1:
                print(f'  Retry {attempt+1}/{retries}: {type(e).__name__}')
                await asyncio.sleep(2)
            else:
                raise


async def list_all_tools():
    async with streamablehttp_client(MCP_URL, headers=HEADERS, timeout=60) as (r, w, _):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            return tools.tools


def pretty(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False))


def extract_data(resp):
    """Extract actual data from the nested data.data structure."""
    if isinstance(resp, dict) and 'data' in resp:
        inner = resp['data']
        if isinstance(inner, dict) and 'data' in inner:
            return inner['data']
        return inner
    return resp


print('OK')

---

## 1. Connect to the MCP Server - List All Available Tools (Free)

First, confirm we can connect, and see what capabilities IO Cloud provides.

In [ ]:
tools = await list_all_tools()

print(f'Found {len(tools)} tools in total:')
print()
for t in tools:
    prefix = '[CaaS]' if t.name.startswith('caas') else '[VMaaS]'
    print(f'  {prefix}  {t.name}')
    print(f'          {t.description}')
    print()

---

## 2. Browse the CaaS Hardware Catalog (Free)

Use `caas_get_hardware_ids` to view currently available GPU models, prices, and inventory.

In [ ]:
caas_hw = await call_tool('caas_get_hardware_ids')
items = extract_data(caas_hw)

# items may be a list or dict; normalize
if isinstance(items, dict):
    items = list(items.values()) if not any(isinstance(v, list) for v in items.values()) else next(v for v in items.values() if isinstance(v, list))

print(f'{len(items)} hardware configurations in total')
print()

# Sort by price
sorted_items = sorted(items, key=lambda x: x.get('price', 999) if isinstance(x, dict) else 999)

print(f'{"GPU":<25} {"hw_id":<8} {"$/hr":<10} {"avail":<8} {"location":<8}')
print('-' * 65)
for it in sorted_items[:15]:
    if isinstance(it, dict):
        print(f'{str(it.get("hardware_name","?")):<25} '
              f'{str(it.get("hardware_id","?")):<8} '
              f'${it.get("price",0):<9.2f} '
              f'{str(it.get("available","?")):<8} '
              f'{str(it.get("location","-")):<8}')

---

## 3. Estimate Deployment Price (Free)

Before spending any money, use `caas_get_price_estimate` to calculate the cost.

We choose RTX 4090 (hw_id=12), in the US (location_id=2), 1 GPU, 1 replica, 1 hour.

In [ ]:
# RTX 4090: hardware_id=12, US: location_id=2
# These are verified available parameters
HW_ID = 12       # GeForce RTX 4090, ~$0.30/hr
LOCATION_ID = 2  # United States

price_est = await call_tool('caas_get_price_estimate', {
    'location_ids': [LOCATION_ID],
    'hardware_id': HW_ID,
    'duration_hours': 1,
    'gpus_per_container': 1,
    'replica_count': 1
})

print('Price estimate:')
pretty(price_est)

---

## 4. Deploy a Container (Costs Money!)

Now let's actually deploy! Configuration:
- **Hardware**: RTX 4090 (hw_id=12)
- **Image**: `nginx:latest`
- **Specs**: 1 GPU, 1 replica, 1 hour

> **Running this cell will incur real charges**. Make sure to execute the "Destroy Deployment" step afterwards!

In [ ]:
deployment_name = f'tutorial-demo-{int(time.time()) % 100000}'

deploy_result = await call_tool('caas_deploy_container', {
    'request': {
        'resource_private_name': deployment_name,
        'duration_hours': 1,
        'gpus_per_container': 1,
        'hardware_id': HW_ID,
        'replica_count': 1,
        'traffic_port': 80,
        'image_url': 'nginx:latest',
        'location_ids': [LOCATION_ID]
    }
})

print(f'Deployment request sent! Name: {deployment_name}')
print()
pretty(deploy_result)

# Extract deployment_id
deployment_id = None
dep_data = extract_data(deploy_result)
if isinstance(dep_data, dict):
    deployment_id = dep_data.get('id') or dep_data.get('deployment_id')
elif isinstance(deploy_result, dict):
    deployment_id = deploy_result.get('id') or deploy_result.get('deployment_id')

print(f'\nDeployment ID: {deployment_id}')
print('(Remember this ID - you will need it to check status and destroy the deployment)')

---

## 5. Check Deployment Status & Container Details (Free)

After submitting the deployment, the container needs some time to start. Let's check the status.

In [ ]:
if deployment_id:
    status = await call_tool('caas_get_deployment', {
        'deployment_id': str(deployment_id)
    })
    print(f'Status of deployment {deployment_id}:')
    print()
    pretty(status)
else:
    print('deployment_id not found. Please check the deployment result from the previous step.')
    print('You can set it manually: deployment_id = "your-ID"')

In [ ]:
# View container/worker details
if deployment_id:
    containers = await call_tool('caas_get_deployment_containers', {
        'deployment_id': str(deployment_id)
    })
    print(f'Container list for deployment {deployment_id}:')
    print()
    pretty(containers)

---

## 6. List All Deployments (Free)

View all CaaS deployments under your account.

In [ ]:
all_deployments = await call_tool('caas_list_deployments', {
    'page': 1,
    'page_size': 5
})

print('All my CaaS deployments:')
print()
pretty(all_deployments)

---

## 7. Destroy the Deployment (Save Money!)

> **Important**: Make sure to destroy the deployment after completing the tutorial, otherwise you will continue to be charged until `duration_hours` expires!

In [ ]:
if deployment_id:
    destroy_result = await call_tool('caas_destroy_deployment', {
        'deployment_id': str(deployment_id)
    })
    print(f'Destroying deployment {deployment_id}:')
    print()
    pretty(destroy_result)
else:
    print('No deployment_id found.')
    print('If you know the ID, you can set it manually:')
    print('deployment_id = "your-ID"  # Fill in and re-run this cell')

In [ ]:
# Confirm destruction
if deployment_id:
    final_status = await call_tool('caas_get_deployment', {
        'deployment_id': str(deployment_id)
    })
    print('Status after destruction:')
    print()
    pretty(final_status)

---

## Summary

This tutorial demonstrated the core features of the IO Cloud MCP server:

| Tool | Purpose |
|------|---------|
| `caas_get_hardware_ids` | View CaaS supported hardware types and prices |
| `caas_get_price_estimate` | Estimate price before deployment |
| `caas_deploy_container` | Deploy a container cluster |
| `caas_get_deployment` | View detailed status of a single deployment |
| `caas_get_deployment_containers` | View containers/workers within a deployment |
| `caas_list_deployments` | List all deployments |
| `caas_destroy_deployment` | Destroy a deployment and stop billing |

### Using with AI Agents

All of the above operations can be performed by AI agents through natural language, for example:

```
"Find the cheapest 4-GPU H100 cluster and deploy my PyTorch image"
"List all my currently running containers"
"Destroy the tutorial-demo deployment"
```